# Projeto final INF <numero-disciplina>

Neste projeto final iremos criar um classificador de insetos para tentar obter a espécie ou pelo menos o gênero de uma instância.

Vamos utilizar datasets coletados manualmente e datasets coletados via Kaggle e Zenodo. 

Datasets:
- João✅
  - 454 imagens
  - 124 classes
- [Insect Identification in the Wild: The AMI Dataset](https://zenodo.org/records/12554005)
  - expert-annotated dataset of 2,893 insect camera trap images (representing 52,948 labeled insects) collected from a global network of automated camera traps, designed to test in-the-wild performance.
- [InsectDCT: Datasets for training and evaluating pipeline processing insect camera recordings](https://zenodo.org/records/21154490?preview_file=classifierTrainV6.zip)🟨
  - 90.591 imagens
  - 207 classes
- [Dangerous Farm Insects Dataset](https://www.kaggle.com/datasets/tarundalal/dangerous-insects-dataset)✅
  - 1.606 imagens
  - 15 classes
- [Insect Images With Scientific Names](https://www.kaggle.com/datasets/shameinew/insect-images-with-scientific-names)
  - 122.665 imagens
  - 2.273 classes
- [IP102-Dataset](https://www.kaggle.com/datasets/rtlmhjbn/ip02-dataset)✅
  - 75.222 imagens
  - 102 classes
- [Insect identification from habitus images](https://www.kaggle.com/datasets/kmldas/insect-identification-from-habitus-images)✅
  - 63.655 imagens
  - 291 classes

Total de imagens sem filtrar/mesclar: 407.141
Total de classes sem filtrar/mesclar: 3.012

Em um primeiro momento iremos padronizar os nomes de cada inseto com o padrão BIN do BOLD Data Portal. Em seguida vamos unificar as classes e enriquecer as classes obtidas com as imagens do BOLD Data Portal.

Posteriormente vamos filtrar as classes com a seguinte regra:
- classes obtidas pelo João entram (independente da quantidade de imagens).
- classes que não são latino americanas.
- classes com menos imagens saem (pensar em algo como tirar as classes com menos de dois desvios padrão da quantidade de imagens entre as classes).

Total de imagens obtidas:
Total de classes obtidas:
Média de imagens por classe:
Mediana de imagens por classe:
Desvio padrão de imagens por classe:
Min:
Max:


Com as classes fechadas, vamos tratar os dados. Temos 3 possíveis problemas: 
- fotos por microscópio tem cerca de 40% da imagem da tela preta (o que está fora do foco). Solução: algorítmo que corta a imagem (o mesmo deve estar presente no pipeline)
- fotos com insetos em folhas nos dados de treino (cor pode esconder o inseto)
- algumas fotos são grandes, necessário reduzir o tamanho
  
Tratados os problemas, vamos pensar nos algorítmos para as classificações. Tenho em mente 3 abordagens:
- Aplicar no Yolo e fazer fine tunning
- Aplicar em modelos pré-treinados do hugging-face e fazer fine tunning
- Hierarchical Mixture of Experts

A depender do tempo, testar o máximo de alternativas possíveis e compará-las.

In [60]:
#imports
import requests
import os 
from typing import Any, Dict, List, Optional, Union
import json
import re
import shutil
import time

In [72]:
# endpoints importantes:

# realiza request
def realiza_request(
    url: str
) -> Optional[Union[Dict[str, Any], List[Any]]]:
    """Realiza uma requisição HTTP GET solicitando payload em formato JSON.

    Args:
        url (str): Endereço do recurso HTTP/HTTPS a ser consultado.

    Returns:
        Optional[Union[Dict[str, Any], List[Any]]]: Conteúdo JSON desserializado
            em caso de sucesso (status 2xx), ou None se houver falha.
    """
    headers = {"accept": "application/json"}
    error_map = {
                400: "Bad Request: Parâmetros ou estrutura de requisição inválidos.",
                401: "Unauthorized: Autenticação necessária.",
                403: "Forbidden: Acesso negado ao recurso.",
                404: "Not Found: O recurso solicitado não existe.",
                500: "Internal Server Error: Erro interno no servidor.",
                502: "Bad Gateway: Resposta inválida do servidor upstream.",
                503: "Service Unavailable: Serviço temporariamente indisponível.",
            }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()

    except requests.exceptions.HTTPError as http_err:
        status_code = response.status_code
        message = error_map.get(
            status_code, f"Erro HTTP não mapeado ({status_code}): {http_err}"
        )
        print("Falha de requisição para '%s': %s", url, message)
        time.sleep(60)

    except requests.exceptions.Timeout:
        print("Tempo limite de requisição excedido para '%s'.", url)

    except requests.exceptions.ConnectionError:
        print("Falha na conexão de rede ao tentar acessar '%s'.", url)

    except requests.exceptions.RequestException as err:
        print("Erro inesperado na comunicação com '%s': %s", url, err)

    return None

# gemini me ajudou aqui, achei chique
def get_lowest_taxonomic_value(data: Dict[str, Optional[str]]) -> Optional[str]:
    """Retorna o valor do nível taxonômico mais específico que não seja nulo.
    Cria uma lista de prioridades ordenados pelo valor do dado e pelo rank"""
    TAXONOMIC_HIERARCHY = ["species", "genus", "tribe", "subfamily", "family"]
    return next(
        (data[rank] for rank in TAXONOMIC_HIERARCHY if data.get(rank)),
        None
    )

# api/query/preprocessor
def bold_data_preprocessor(
        termo:str
) -> str :
    '''Endpoint para gerar o triplet_token a partir de um termo
    Args:
        termo (str): Nome da espécie a ser parseada.
    
    Returns:
        str: token parseado para obter o query_id
    '''

    url = f'https://portal.boldsystems.org/api/query/preprocessor?query={termo}'

    response = realiza_request(url)
    triplet_token = response['successful_terms'][0]['matched']

    return triplet_token

# api/query
def bold_data_query(
        triplet_token:str
) -> str :
    '''Endpoint para gerar o query_id a partir do termo
    Args:
        triplet_token (str): token usado para obter o query_id.
    
    Returns:
        str: o query_id
    '''

    url = f'https://portal.boldsystems.org/api/query?query={triplet_token}&extent=limited'

    response = realiza_request(url)
    query_id = response['query_id']

    return query_id

class QueryVazia(Exception):
    """Excessão lançada normalmente quando não é encontrado o nome da espécie"""
    pass

# api/documents/
def bold_data_documents(
        query_id:str
) -> str:
    '''Endpoint que pode retornar o species
    Args:
        query_id (str): id do inseto obtido via bold_data_query().
    
    Returns:
        str: nome da espécie encontrada
    '''

    url = f'https://portal.boldsystems.org/api/documents/{query_id}?length=1&start=0'

    response = realiza_request(url)
    if not response['data']:
        raise QueryVazia("Resposta de documents veio vazio, nome de pasta errado")
    dados = {
        "species":response['data'][0]['species'],
        "genus":response['data'][0]['genus'],
        "tribe":response['data'][0]['tribe'],
        "subfamily":response['data'][0]['subfamily'],
        "family":response['data'][0]['family']
    }

    return get_lowest_taxonomic_value(dados)

# api/images/
def bold_data_images(
        query_id:str
) -> list[str]:
    '''Endpoint que retorna o link para baixar imagens
    Args:
        query_id (str): id do inseto obtido via bold_data_query().
    
    Returns:
        str: lista dos links para download das imagens
    '''
    image_urls = list()
    url = f'https://portal.boldsystems.org/api/images/{query_id}?max_images=-1'

    response = realiza_request(url)
    for image in response['images']:
        image_urls.append(image["image_url"])

    return image_urls

# Usando API GBIF para mapear nomes científicos
def gbif_insect_names(
        id:str
) -> str:
    "Função que mapeia id pelo nome dos insetos"
    url = f'https://api.gbif.org/v1/species/{id}'

    response = realiza_request(url)
    if not response:
        raise QueryVazia("Resposta de documents veio vazio, nome de pasta errado")
    dados = {
        "species":response['species'],
        "genus":response['genus'],
        "tribe":response['order'],
        "subfamily":response['phylum'],
        "family":response['family']
    }

    return get_lowest_taxonomic_value(dados)

## Tratando arquivos do João

In [34]:
ja_passou = list()
com_problema = list()

In [45]:
def get_nome_pasta(
        termo:str
) -> str:
    """Função que busca no BALD Data Portal o nome científico dado nome da pasta"""
    termo = re.sub(r'sp\d+', '',termo.lower()).strip().title()
    if ' ' in termo:
        termo = termo.split()[-1].strip()

    # print(f"termo: {termo}")
    triplet_token = bold_data_preprocessor(termo)
    # print(f'triplet_token: {triplet_token}')
    query_id = bold_data_query(triplet_token)
    # print(f'query_id: {query_id}')
    specie = bold_data_documents(query_id)
    # print(f'specie: {specie}')
    return specie

def get_nome_pasta_v2(
        termo:str
) -> str:
    """Função que busca no BALD Data Portal o nome científico dado nome da pasta"""
    termo = re.sub(r'sp\d+|_fw||\([^)]*\)', '',termo.lower()).strip().capitalize()
    if ' ' in termo:
        termo = termo.split()
        # if len(termo) >= 4:
        if len(termo) == 2:
            termo = ' '.join(termo)
            termo = "tax:species:"+termo.capitalize()
        elif len(termo) == 3:
            termo = ' '.join(termo[-3:-1])
            termo = "tax:species:"+termo.capitalize()
        else: termo = termo[-1].strip().capitalize()

    try:
        # print(f"termo: {termo}")
        triplet_token = bold_data_preprocessor(termo)
        # print(f'triplet_token: {triplet_token}')
        query_id = bold_data_query(triplet_token)
        # print(f'query_id: {query_id}')
        specie = bold_data_documents(query_id)
        # print(f'specie: {specie}')
    except Exception as e:
        print(e)
        print(f"termo: {termo}")
        specie = ''
    return specie


def lista_pastas(
        caminho: str
) -> list[str]:
    """Função que dado um caminho coloca o nome das pastas nele em uma lista"""
    return os.listdir(caminho)

In [ ]:

try:
    diretorio='/home/bradachi/Downloads/datasets/Insetos taxo'
    diretorio_destino='/home/bradachi/Downloads/datasets/final'
    pastas = lista_pastas(diretorio)
    faltam = len(pastas)
    atual = 1

    for pasta in pastas:
        print(f'[{atual}/{faltam}]')

        atual+=1

        if pasta in ja_passou:
            continue

        nome_novo = get_nome_pasta(pasta)
        origem = f'{diretorio}/{pasta}'
        destino = f'{diretorio_destino}/{nome_novo}'

        shutil.copytree(origem, destino, dirs_exist_ok=True)

        ja_passou.append(pasta)

except Exception as e:
    print(e)
    print(pasta)

[1/123]
[2/123]
[3/123]
[4/123]
[5/123]
[6/123]
[7/123]
[8/123]
[9/123]
[10/123]
[11/123]
[12/123]
[13/123]
[14/123]
[15/123]
[16/123]
[17/123]
[18/123]
[19/123]
[20/123]
[21/123]
[22/123]
[23/123]
[24/123]
[25/123]
[26/123]
[27/123]
[28/123]
[29/123]
[30/123]
[31/123]
[32/123]
[33/123]
[34/123]
[35/123]
[36/123]
[37/123]
[38/123]
[39/123]
[40/123]
[41/123]
[42/123]
[43/123]
[44/123]
[45/123]
[46/123]
[47/123]
[48/123]
[49/123]
[50/123]
[51/123]
[52/123]
[53/123]
[54/123]
[55/123]
[56/123]
[57/123]
[58/123]
[59/123]
[60/123]
[61/123]
[62/123]
[63/123]
[64/123]
[65/123]
[66/123]
[67/123]
[68/123]
[69/123]
[70/123]
[71/123]
[72/123]
[73/123]
[74/123]
[75/123]
[76/123]
[77/123]
[78/123]
[79/123]
[80/123]
[81/123]
[82/123]
[83/123]
[84/123]
[85/123]
[86/123]
[87/123]
[88/123]
[89/123]
[90/123]
[91/123]
[92/123]
[93/123]
[94/123]
[95/123]
[96/123]
[97/123]
[98/123]
[99/123]
[100/123]
[101/123]
[102/123]
[103/123]
[104/123]
[105/123]
[106/123]
[107/123]
[108/123]
[109/123]
[110/123]
[111/123

## Tratando arquivos Dangerous Farm Insects Dataset

In [46]:
try:
    diretorio='/home/bradachi/Downloads/datasets/farm_insects'
    diretorio_destino='/home/bradachi/Downloads/datasets/final'
    pastas = lista_pastas(diretorio)
    faltam = len(pastas)
    atual = 1

    for pasta in pastas:
        print(f'[{atual}/{faltam}]')

        atual+=1

        if pasta in ja_passou:
            continue

        # nome_novo = get_nome_pasta(pasta)
        origem = f'{diretorio}/{pasta}'
        destino = f'{diretorio_destino}/{pasta}'

        shutil.copytree(origem, destino, dirs_exist_ok=True)

        ja_passou.append(pasta)

except Exception as e:
    print(e)
    print(pasta)

[Errno 2] No such file or directory: '/home/bradachi/Downloads/datasets/farm_insects'
15


## Tratando arquivos Insect identification from habitus images

In [15]:
ja_passou = list()

In [16]:

try:
    diretorio='/home/bradachi/Downloads/datasets/database'
    diretorio_destino='/home/bradachi/Downloads/datasets/final'
    # diretorio='/home/bradachi/Downloads/datasets/pastateste'
    # diretorio_destino='/home/bradachi/Downloads/datasets/pastateste (cópia)'
    pastas = lista_pastas(diretorio)
    faltam = len(pastas)
    atual = 1

    for pasta in pastas:
        print(f'[{atual}/{faltam}]')

        atual+=1

        if pasta in ja_passou:
            continue

        nome_novo = gbif_insect_names(pasta)
        origem = f'{diretorio}/{pasta}'
        destino = f'{diretorio_destino}/{nome_novo}'

        shutil.copytree(origem, destino, dirs_exist_ok=True)

        ja_passou.append(pasta)

except Exception as e:
    print(e)
    print(pasta)

[1/291]
[2/291]
[3/291]
[4/291]
[5/291]
[6/291]
[7/291]
[8/291]
[9/291]
[10/291]
[11/291]
[12/291]
[13/291]
[14/291]
[15/291]
[16/291]
[17/291]
[18/291]
[19/291]
[20/291]
[21/291]
[22/291]
[23/291]
[24/291]
[25/291]
[26/291]
[27/291]
[28/291]
[29/291]
[30/291]
[31/291]
[32/291]
[33/291]
[34/291]
[35/291]
[36/291]
[37/291]
[38/291]
[39/291]
[40/291]
[41/291]
[42/291]
[43/291]
[44/291]
[45/291]
[46/291]
[47/291]
[48/291]
[49/291]
[50/291]
[51/291]
[52/291]
[53/291]
[54/291]
[55/291]
[56/291]
[57/291]
[58/291]
[59/291]
[60/291]
[61/291]
[62/291]
[63/291]
[64/291]
[65/291]
[66/291]
[67/291]
[68/291]
[69/291]
[70/291]
[71/291]
[72/291]
[73/291]
[74/291]
[75/291]
[76/291]
[77/291]
[78/291]
[79/291]
[80/291]
[81/291]
[82/291]
[83/291]
[84/291]
[85/291]
[86/291]
[87/291]
[88/291]
[89/291]
[90/291]
[91/291]
[92/291]
[93/291]
[94/291]
[95/291]
[96/291]
[97/291]
[98/291]
[99/291]
[100/291]
[101/291]
[102/291]
[103/291]
[104/291]
[105/291]
[106/291]
[107/291]
[108/291]
[109/291]
[110/291]
[111/291

## Tratando IP102 Dataset


In [18]:
ja_passou=list()


In [19]:
# vamos nomear corretamente as pastas
dicionario = dict()
with open('/home/bradachi/Downloads/datasets/archive(1)/classes.txt', 'r', encoding='utf-8') as arquivo:
    for linha in arquivo:
        linha = linha.split()
        dicionario[str(int(linha[0])-1)] = ' '.join(linha[1:])

# diretorio='/home/bradachi/Downloads/datasets/archive(1)/classification/train'
diretorio='/home/bradachi/Downloads/datasets/archive(1)/classification/test'
diretorio_destino='/home/bradachi/Downloads/datasets/final'
pastas = lista_pastas(diretorio)
faltam = len(pastas)
atual = 1

for pasta in pastas:
    print(f'[{atual}/{faltam}]')

    atual+=1

    if pasta in ja_passou:
        continue

    nome_novo = dicionario[pasta]
    origem = f'{diretorio}/{pasta}'
    destino = f'{diretorio_destino}/{nome_novo}'

    shutil.copytree(origem, destino, dirs_exist_ok=True)

    ja_passou.append(pasta)

[1/102]
[2/102]
[3/102]
[4/102]
[5/102]
[6/102]
[7/102]
[8/102]
[9/102]
[10/102]
[11/102]
[12/102]
[13/102]
[14/102]
[15/102]
[16/102]
[17/102]
[18/102]
[19/102]
[20/102]
[21/102]
[22/102]
[23/102]
[24/102]
[25/102]
[26/102]
[27/102]
[28/102]
[29/102]
[30/102]
[31/102]
[32/102]
[33/102]
[34/102]
[35/102]
[36/102]
[37/102]
[38/102]
[39/102]
[40/102]
[41/102]
[42/102]
[43/102]
[44/102]
[45/102]
[46/102]
[47/102]
[48/102]
[49/102]
[50/102]
[51/102]
[52/102]
[53/102]
[54/102]
[55/102]
[56/102]
[57/102]
[58/102]
[59/102]
[60/102]
[61/102]
[62/102]
[63/102]
[64/102]
[65/102]
[66/102]
[67/102]
[68/102]
[69/102]
[70/102]
[71/102]
[72/102]
[73/102]
[74/102]
[75/102]
[76/102]
[77/102]
[78/102]
[79/102]
[80/102]
[81/102]
[82/102]
[83/102]
[84/102]
[85/102]
[86/102]
[87/102]
[88/102]
[89/102]
[90/102]
[91/102]
[92/102]
[93/102]
[94/102]
[95/102]
[96/102]
[97/102]
[98/102]
[99/102]
[100/102]
[101/102]
[102/102]


## Tratando InsectDCT: Datasets for training and evaluating pipeline processing insect camera recordings

In [6]:
ja_passou = list()

In [71]:
# solução semelhante ao do Joao

def mescla_dados(diretorio_origem, diretorio_destino):
    try:
        pastas = lista_pastas(diretorio_origem)
        faltam = len(pastas)
        atual = 1

        for pasta in pastas:
            print(f'[{atual}/{faltam}]')

            atual+=1

            if pasta in ja_passou:
                continue
            
            nome_novo = get_nome_pasta_v2(pasta)

            if not nome_novo:
                print(f'Erro em {pasta}, salvando e continuando')
                com_problema.append(pasta)
                continue

            origem = f'{diretorio_origem}/{pasta}'
            destino = f'{diretorio_destino}/{nome_novo}'

            shutil.copytree(origem, destino, dirs_exist_ok=True)

            ja_passou.append(pasta)

    except Exception as e:
        print(e)
        print(pasta)

def log_erros():
    print("salvando log de erro")
    with open("log.txt", "w", encoding="utf-8") as arquivo:
        for pasta in com_problema:
            arquivo.write(pasta+'\n')


In [15]:
# diretorio_destino='/home/bradachi/Downloads/datasets/pastateste (cópia)'
# diretorio_origem='/home/bradachi/Downloads/datasets/pastateste'

diretorio_destino='/home/bradachi/Downloads/datasets/final'
diretorio_origem='/home/bradachi/Downloads/datasets/datasetV6/GBIF'

#pesquisa por dois termos
mescla_dados(diretorio_origem, diretorio_destino)


[1/51]
[2/51]
[3/51]
[4/51]
[5/51]
[6/51]
[7/51]
[8/51]
[9/51]
[10/51]
[11/51]
[12/51]
[13/51]
[14/51]
[15/51]
[16/51]
[17/51]
[18/51]
[19/51]
[20/51]
[21/51]
[22/51]
[23/51]
[24/51]
[25/51]
[26/51]
[27/51]
[28/51]
[29/51]
[30/51]
[31/51]
[32/51]
[33/51]
[34/51]
[35/51]
[36/51]
[37/51]
[38/51]
[39/51]
[40/51]
[41/51]
[42/51]
[43/51]
[44/51]
[45/51]
[46/51]
[47/51]
[48/51]
[49/51]
[50/51]
[51/51]


In [17]:
diretorio_destino='/home/bradachi/Downloads/datasets/final'
diretorio_origem='/home/bradachi/Downloads/datasets/datasetV6/NI'

#pesquisa por dois termos
mescla_dados(diretorio_origem, diretorio_destino)

[1/10]
[2/10]
[3/10]
[4/10]
[5/10]
[6/10]
[7/10]
[8/10]
[9/10]
[10/10]


In [32]:
diretorio_destino='/home/bradachi/Downloads/datasets/final'
diretorio_origem='/home/bradachi/Downloads/datasets/datasetV6/NI2'

#pesquisa por dois termos
mescla_dados(diretorio_origem, diretorio_destino)

[1/63]
[2/63]
[3/63]
[4/63]
[5/63]
[6/63]
[7/63]
[8/63]
[9/63]
[10/63]
[11/63]
[12/63]
[13/63]
[14/63]
[15/63]
[16/63]
[17/63]
[18/63]
[19/63]
[20/63]
[21/63]
[22/63]
[23/63]
[24/63]
[25/63]
[26/63]
[27/63]
[28/63]
[29/63]
[30/63]
[31/63]
[32/63]
[33/63]
[34/63]
[35/63]
[36/63]
[37/63]
[38/63]
[39/63]
[40/63]
[41/63]
[42/63]
[43/63]
[44/63]
[45/63]
[46/63]
[47/63]
[48/63]
[49/63]
[50/63]
[51/63]
[52/63]
[53/63]
[54/63]
[55/63]
[56/63]
[57/63]
[58/63]
[59/63]
[60/63]
[61/63]
[62/63]
[63/63]


In [35]:
diretorio_destino='/home/bradachi/Downloads/datasets/final'
diretorio_origem='/home/bradachi/Downloads/datasets/datasetV6/NI2_MAMBO'

#pesquisa por dois termos
mescla_dados(diretorio_origem, diretorio_destino)

[1/12]
[2/12]
[3/12]
[4/12]
[5/12]
[6/12]
[7/12]
[8/12]
[9/12]
[10/12]
[11/12]
[12/12]


In [13]:
diretorio_destino='/home/bradachi/Downloads/datasets/final'
diretorio_origem='/home/bradachi/Downloads/datasets/datasetV6/Orchard'

#pesquisa por dois termos
mescla_dados(diretorio_origem, diretorio_destino)

[1/55]
[2/55]
[3/55]
[4/55]
[5/55]
[6/55]
[7/55]
[8/55]
[9/55]
[10/55]
[11/55]
[12/55]
[13/55]
[14/55]
[15/55]
[16/55]
[17/55]
[18/55]
[19/55]
[20/55]
[21/55]
[22/55]
[23/55]
[24/55]
[25/55]
[26/55]
[27/55]
[28/55]
[29/55]
[30/55]
[31/55]
[32/55]
[33/55]
[34/55]
[35/55]
[36/55]
[37/55]
[38/55]
[39/55]
[40/55]
[41/55]
[42/55]
[43/55]
[44/55]
[45/55]
[46/55]
[47/55]
[48/55]
[49/55]
[50/55]
[51/55]
[52/55]
[53/55]
[54/55]
[55/55]


## Tratando Insect Images With Scientific Names

In [73]:
ja_passou = list()
com_problema = list()

In [74]:
diretorio_origem='/home/bradachi/Downloads/datasets/archive/insects'
diretorio_destino='/home/bradachi/Downloads/datasets/final'

# diretorio_destino='/home/bradachi/Downloads/datasets/pastateste (cópia)'
# diretorio_origem='/home/bradachi/Downloads/datasets/pastateste'

#pesquisa por dois termos
mescla_dados(diretorio_origem, diretorio_destino)

log_erros() # lidar com falha no servidor.

[1/2258]
[2/2258]
[3/2258]
Resposta de documents veio vazio, nome de pasta errado
termo: tax:species:Dolichotetranychus floridanus
Erro em Dolichotetranychus floridanus (Banks), salvando e continuando
[4/2258]
[5/2258]
[6/2258]
[7/2258]
[8/2258]
Resposta de documents veio vazio, nome de pasta errado
termo: tax:species:Anomala oblivia
Erro em Anomala oblivia Horn, salvando e continuando
[9/2258]
[10/2258]
[11/2258]
Resposta de documents veio vazio, nome de pasta errado
termo: tax:species:Tetranychus canadensis
Erro em Tetranychus canadensis (McGregor), salvando e continuando
[12/2258]
[13/2258]
[14/2258]
[15/2258]
Falha de requisição para '%s': %s https://portal.boldsystems.org/api/documents/eAErSaywKi5ITc5MLbbyT0otSk1UKCnKLCjNSy5JLEm0zsnMzSxJTQEAIREO7g==?length=1&start=0 Service Unavailable: Serviço temporariamente indisponível.
'NoneType' object is not subscriptable
termo: tax:species:Oberea tripunctata
Erro em Oberea tripunctata (Swederus), salvando e continuando
[16/2258]
[17/2258]


In [ ]:
# arrumar as 700 que deu erro


## Insect Identification in the Wild: The AMI Dataset